# Assignment 3, Problem 2: Genetic Algorithm for Cell Tower Placement

This notebook implements a genetic algorithm (GA) for the cell tower problem from Assignment 2.
Most of the GA is written for you. Your job is to complete the two functions in **Part 1**, and then
run the experiments in **Part 3**.

Put `city_adjacency.csv` in the same folder as this notebook.

**Representation.** A solution is a **chromosome**: a list of 25 genes, one per region, where gene
`j` is 1 if a tower is built in region `j + 1` and 0 otherwise. (Python counts from 0, so position
`j` holds region `j + 1`.)

**Cost.** The GA minimizes

$$\text{cost} = (\text{number of towers}) + \text{penalty} \times (\text{number of uncovered regions}).$$

A region is **covered** if it has a tower, or if one of its adjacent regions has a tower.

In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt

adj = pd.read_csv("city_adjacency.csv", index_col=0)   # the first column holds the region labels
A = adj.values                                        # A[i][j] = 1 if regions i+1 and j+1 are adjacent
n = len(A)                                            # 25 regions, so 25 genes per chromosome
print("regions:", n)

---
## Part 1. Complete the cost and mutation functions

The function `count_uncovered` is provided. Complete `cost` and `mutate`: each says what it should do.
Replace the line `raise NotImplementedError(...)` with your code. Then run the test cell below.

In [ ]:
def count_uncovered(chromosome):
    # PROVIDED. Return the number of regions that are NOT covered by this chromosome.
    # Region i is covered if chromosome[i] == 1, or chromosome[j] == 1 for some j with A[i][j] == 1.
    uncovered = 0
    for i in range(n):
        covered = chromosome[i] == 1 or any(chromosome[j] == 1 and A[i][j] == 1 for j in range(n))
        if not covered:
            uncovered += 1
    return uncovered


def cost(chromosome, penalty):
    # Return (number of towers) + penalty * (number of uncovered regions).
    # The number of towers is the number of genes equal to 1. Use count_uncovered above.
    raise NotImplementedError("Complete cost")


def mutate(chromosome, rate, rng):
    # Return a NEW list. For each gene, in order, draw rng.random(); if it is below `rate`,
    # flip the gene (0 becomes 1, 1 becomes 0). Otherwise keep it.
    # Use rng.random(), not random.random(), so that results can be reproduced with a seed.
    raise NotImplementedError("Complete mutate")

In [ ]:
# Test cell. Each line prints your result next to the value it should have.
def from_regions(regions):
    # Build a chromosome with towers in the given regions (labels 1 to 25).
    return [1 if (j + 1) in regions else 0 for j in range(n)]

no_towers  = [0] * n
all_towers = [1] * n
seven      = from_regions([1, 3, 10, 12, 13, 21, 24])   # an optimal solution from Assignment 2
six        = from_regions([3, 10, 12, 13, 21, 24])      # the same, without the tower in region 1

print("cost, no towers, penalty 5:   ", cost(no_towers, 5), "  should be 125")
print("cost, seven towers, penalty 5:", cost(seven, 5),     "  should be 7")
print("cost, six towers, penalty 5:  ", cost(six, 5),       "  should be 16")
print("cost, six towers, penalty 1:  ", cost(six, 1),       "  should be 8")

rng = random.Random(0)
print("mutate all towers, rate 0.5:", mutate(all_towers, 0.5, rng)[:8], "  should start [1, 1, 0, 0, 1, 0, 1, 0]")

---
## Part 2. The genetic algorithm (provided)

Read through these functions; you do not need to change them. They follow the steps from lecture
(population, selection, crossover, mutation, new generation), with these specific choices:

- **Initial population.** Each gene of each chromosome is 1 with probability `p_one`.
- **Selection (tournament of 2).** To choose a parent, pick two chromosomes at random and keep the
  one with lower cost. This makes fitter chromosomes more likely to become parents: a good
  chromosome wins most of the tournaments it enters, while a poor one rarely does.
- **Crossover (one point).** Pick a cut point `k` at random; child 1 takes genes `0` to `k - 1` from
  parent A and the rest from parent B, and child 2 the reverse.
- **Mutation.** Your `mutate` function, applied to every child.
- **Keeping the best.** The best chromosome found so far is copied unchanged into each new
  generation, so it is never lost. The rest of the new generation consists of children.
- **Stopping rule.** A fixed number of generations.

In [ ]:
def random_population(size, p_one, rng):
    return [[1 if rng.random() < p_one else 0 for _ in range(n)] for _ in range(size)]


def tournament_select(population, penalty, rng):
    a, b = rng.sample(population, 2)
    return a if cost(a, penalty) <= cost(b, penalty) else b


def one_point_crossover(parent_a, parent_b, rng):
    k = rng.randint(1, n - 1)
    return parent_a[:k] + parent_b[k:], parent_b[:k] + parent_a[k:]


def genetic_algorithm(penalty, mutation_rate=1 / 25, pop_size=30, generations=100,
                      p_one=0.3, seed=0):
    rng = random.Random(seed)
    population = random_population(pop_size, p_one, rng)
    best = min(population, key=lambda c: cost(c, penalty))
    history = [cost(best, penalty)]                      # best cost after each generation

    for _ in range(generations):
        new_population = [list(best)]                    # keep the best
        while len(new_population) < pop_size:
            parent_a = tournament_select(population, penalty, rng)
            parent_b = tournament_select(population, penalty, rng)
            for child in one_point_crossover(parent_a, parent_b, rng):
                child = mutate(child, mutation_rate, rng)
                if len(new_population) < pop_size:
                    new_population.append(child)
        population = new_population

        candidate = min(population, key=lambda c: cost(c, penalty))
        if cost(candidate, penalty) < cost(best, penalty):
            best = candidate
        history.append(cost(best, penalty))

    return best, history


def describe(chromosome):
    towers = [j + 1 for j in range(n) if chromosome[j] == 1]
    return {"towers": len(towers), "uncovered": count_uncovered(chromosome), "regions": towers}


def plot_history(history, title):
    plt.figure(figsize=(8, 3.5))
    plt.plot(history)
    plt.xlabel("generation")
    plt.ylabel("best cost so far")
    plt.title(title)
    plt.show()

---
## Part 3. Experiments

Example: one run with penalty 5 and seed 0.

In [ ]:
best, history = genetic_algorithm(penalty=5, seed=0)
print(describe(best))
plot_history(history, "Penalty 5, seed 0")

Use the cells below for parts (b) and (c) of Problem 2. A helper that runs seeds 0 to 9 and
collects the results in a table is provided.

In [ ]:
def run_seeds(penalty, mutation_rate=1 / 25, seeds=range(10)):
    rows = []
    for s in seeds:
        best, _ = genetic_algorithm(penalty=penalty, mutation_rate=mutation_rate, seed=s)
        rows.append({"seed": s, **describe(best)})
    return pd.DataFrame(rows)

# Part (b): run_seeds(5)
# Part (c): run_seeds(5, mutation_rate=0)